# South African Parliament dataset

Now because of my own personal backround of living in this country and the thereby given interest to ivestigate it's parliament, and additionaly it having a really diverse (race and gender) parliament I picked South Africa as further data set to test my classifier. 

The national assembly members can be found on this website: https://www.parliament.gov.za/group-details

Looking into the html structure of the website one can find the following block for each member:

                         <member-detail><div class="ng-binding">                 <div class="member-image">                   <a href="/person-details/5549" ng-show="member.profile_pic_url">                     <img src="/storage/app/media/MemberImages/5549.jpg" alt="">                   </a>                   <div ng-show="!member.profile_pic_url" class="ng-hide"></div>                 </div>                   <a href="/person-details/5549" class="ng-binding">                     Mncedisi Nontsele                   </a>                   <br>                   ANC                   <br>                   <div class="ng-binding">Eastern Cape</div>               </div></member-detail>                       

So we get the member-image, the name, the party and the region quite easily. Now getting the gender is a bit tricky since it is not provided. 

Let's scrape this webiste first and save the meta data to a csv and the images in a seperate folder using beautiful soup. 




In [ ]:
import requests
from bs4 import BeautifulSoup
import csv
import os

# Define the URL of the Parliament page
url = "https://www.parliament.gov.za/group-details"

# Fetch the page content
response = requests.get(url)
soup = BeautifulSoup(response.content, "html.parser")
# soup 


Looking at the soup, I find that there is a json file already containing all member cards.

'div class="cell" id="all-members-tabs-wrapper">
group-details member-count="437" members='{"a-d":[{"id":6067,"full_name":"Zelna Saira Abader","profile_pic_url":"\/storage\/app\/media\/MemberImages\/6067.jpg","party":"MK","province":"Gauteng","national":0},{"id":5164,"full_name":"Alexandra Lilian Amelia Abrahams","profile_pic_url":"\/storage\/app\/media\/MemberImages\/5164.jpg","party":"DA","province":"","national":1}...
/div>'


 Now we just need to guide the extractor there. I gotta admit, GPT helped me to write the piece of code below to neatly extract all the information. 

In [ ]:
import requests
from bs4 import BeautifulSoup
import csv
import json
import os
from urllib.parse import urljoin

# Base URL
base_url = "https://www.parliament.gov.za"

# Define URL for the page
url = urljoin(base_url, "/group-details")

# Fetch page content
response = requests.get(url)
soup = BeautifulSoup(response.content, "html.parser")

# Locate the JSON-like data within the 'group-details' tag
members_data = soup.find("group-details")['members']
members_dict = json.loads(members_data)  # Parse the JSON-like string into a dictionary

# Create directory for images if it doesn't exist
os.makedirs("../datasets/images", exist_ok=True)

# Open CSV file to write the extracted data
with open("../datasets/members.csv", mode="w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["ID", "Name", "Party", "Province"])

    # Iterate over each member, saving data to CSV and images
    for key, member_list in members_dict.items():
        for member in member_list:
            member_id = member['id']
            full_name = member['full_name']
            party = member['party']
            province = member['province'] or "N/A"

            # Write to CSV
            writer.writerow([member_id, full_name, party, province])

            # Download and save image
            img_url = urljoin(base_url, member['profile_pic_url'].replace("\\", ""))
            img_data = requests.get(img_url).content
            img_filename = f"../datasets/images/{member_id}.jpg"
            with open(img_filename, "wb") as img_file:
                img_file.write(img_data)

            print(f"Processed member: {full_name}, ID: {member_id}")

print("Data extraction complete. Check '../datasets/members.csv' and '../datasets/images' for results.")


Now to identify the gender of each member, I asked gender_guesser to label them. Since there are a lot of uncommon African Names this resulted in many unknowns.

In [ ]:
import pandas as pd
import gender_guesser.detector as gender

# Load the CSV file
df = pd.read_csv('../datasets/members.csv')

# Initialize the gender detector
d = gender.Detector()

# Function to predict gender based on the first name
def predict_gender(name):
    first_name = name.split()[0]
    gender = d.get_gender(first_name)
    if gender in ['male', 'mostly_male']:
        return 'Male'
    elif gender in ['female', 'mostly_female']:
        return 'Female'
    else:
        return 'Unknown'

# Add the gender column
df['Gender'] = df['Name'].apply(predict_gender)

# Save the updated DataFrame to a new CSV file
df.to_csv('../datasets/members_with_gender.csv', index=False)

# Display the first few rows of the updated DataFrame
print(df.head())

The script takes about 20 minutes to run and loads a total of 438 members.

Cabinet, speakers and 400 parliament seats: 

https://en.wikipedia.org/wiki/National_Assembly_of_South_Africa#Current_composition_of_Parliament_compared_to_%_of_vote

Load the South African member metadata and images using the helper classes below. 

In [ ]:
class Diputado(object):
    def __init__(self,ind,fileRoot="./"):
        self.name=""
        # self.surname=""
        self.ind=ind
        self.picfile=""
        self.party=""
        self.gender=""
        self.fileRoot=fileRoot
        self.ext=".jpg" # note removed the c 
    def setName(self,name):
        self.name=name
    def setProvince(self,province):
        self.province=province
    def setParty(self,party):
        self.party=party
    def setGender(self,gender):
        self.gender=gender
    def getName(self):
        return self.name
    def getProvince(self):
        return self.province
    def getInd(self):
        return self.ind
    def getPicfile(self):
        return self.fileRoot + str(self.ind) + self.ext
    def getParty(self):
        return self.party
    def getGender(self):
        return self.gender
       

class Parlament(object):
    def __init__(self):
        self.elements=[]
        self.inds=[]
        self.ndips=0
    def add(self, diputado):
        self.elements.append(diputado)
        self.inds.append(diputado.getInd())
        self.ndips +=1
    def getInds(self):
        return [self.elements[i].getInd() for i in range(self.ndips)]
    def len(self):
        return len(self.elements)
    def __getitem__(self, key):
        if isinstance(key, slice):
            indices = key.indices(self.ndips)
            return [self[ii] for ii in range(*key.indices(self.len()))] 
        else:
            return self.elements[key]
    def getName(self,key):
        return self.elements[key].getName()
    

import csv
with open('../datasets/members_with_gender.csv','r', encoding="utf8") as csvfile:
    r = csv.reader(csvfile, delimiter=',')
    next(r)
    p = Parlament()
    for row in r:
        n = Diputado(int(row[0]),"../datasets/images/")
        n.setName(row[1].strip())
        n.setParty(row[2].strip())
        n.setProvince(row[3].strip())
        n.setGender(row[4].strip()) #note changed indexes
        p.add(n)

print(p.len())
print(p[99].getPicfile())
print(p[99].getGender())

Unfortunately here I ran out of time and had to submit :(

In [ ]:
from scipy import ndimage,misc
import numpy as np
import matplotlib.pyplot as pl
idip = 99
I = pl.imread(p[idip].getPicfile())
pl.imshow(I)
np.shape(I)
' | '.join([p[idip].getName(), p[idip].getGender()])

In [ ]:
print(len(pl.imread(p[idip].getPicfile()).mean(axis=2)))
I_lum = I[: ,: , 2]
pl.imshow(I_lum)
I[: ,: , 2]


In [ ]:
X=np.array([ pl.imread(d.getPicfile()).mean(axis=2).ravel()
             for d in p])
Yg = np.array([ d.getGender() for d in p])
Yp = np.array([ d.getParty() for d in p])
Yp[0:5],Yg[0:5],X.shape